# 02 — The Equation Space Engine

**GenerationalLineage companion notebook.** Toolset: `engine/toolsets/equation_space.py`.

Steering through a parametrized family of equations by following a
"collapse function" `rho(s) >= 0`'s own gradient, rather than searching
blind. `descend()` reads `rho` and its gradient at one point (free);
`build_up()` walks from a start point to `rho=0` by gradient descent
(cost = steps, genuinely refuses if the walk stalls).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from engine.toolsets import equation_space as eqs
from engine.lines import AscentNotFree

d = eqs.descend(0.5 + 0.1j)
print(d)


{'toolset': 'equation_space', 's': (0.5+0.1j), 'rho': 0.415929203539823, 'gradient': (-0.9945962926927177+0.7596522801250893j), '|gradient|': 1.2515164290321343, 'note': 'one evaluation + a local finite-difference gradient — the free reading'}


## build_up — walking to the locus, and refusing when it can't

In [2]:
b = eqs.build_up({"start": 0.5+0.1j})
print(b)
print(f"distance from s=i: {abs(b['s']-1j):.6f}  (expected sqrt(2)={2**0.5:.6f})")

try:
    eqs.build_up({"start": 100+100j}, max_steps=2)
    print("did NOT refuse -- unexpected")
except AscentNotFree as e:
    print(f"refused, as it should: {e}")


{'toolset': 'equation_space', 'converged': True, 's': (0.7843000059066828-0.1766768545756966j), 'rho': 9.489462467697496e-05, 'cost': 19}
distance from s=i: 1.414106  (expected sqrt(2)=1.414214)
refused, as it should: did not converge; best s=(100+100j), rho=0.98009999504975


## classify_singularity — fold (caustic) vs smooth minimum

In [3]:
fold = eqs.classify_singularity(b["s"])
print(fold)


{'toolset': 'equation_space', 's0': (0.7843000059066828-0.1766768545756966j), 'ratios': {-0.03: 0.4981914968230935, -0.01: 0.499820864597926, 0.01: 0.4762294225143418, 0.03: 0.47802552855945085}, 'is_fold_caustic': True, 'note': 'ratios converging to the same nonzero constant from both sides = a fold (A2 caustic, the equation genuinely collapses there); converging to 0 = a smooth minimum (rho merely gets small)'}


## steering_correlation — does Gamma's own curvature predict the costly gradient?

In [4]:
corr = eqs.steering_correlation(eqs._gamma_curvature)
print(corr)


{'toolset': 'equation_space', 'n': 150, 'pearson': 0.9733640954093495}


## The cross-check this notebook exists for

The circle found here (`center=i, radius=sqrt(2)`) by pure gradient
descent, with no sedenion machinery at all, should match the circle
found independently in `SedenionSpectralRelativity/
prime_gauge_sedenion.py` via an actual sedenion zero-divisor
construction. Two unrelated methods, same answer.

In [5]:
import json
print(json.dumps(eqs.verify(), indent=2, default=str))


{
  "ok": true,
  "ok_descend": true,
  "ok_converge": true,
  "ok_matches_known_circle": true,
  "converged_distance_from_i": 1.4141056959645475,
  "ok_refuses_when_unreachable": true,
  "ok_fold_caustic": true,
  "steering_correlation_pearson": 0.9733640954093495,
  "ok_correlation_positive": true
}
